In [ ]:
from IPython import display
import time
import torch

In [ ]:
#Custom Gaussian activation function
class GaussianActivation(torch.nn.Module):
    def __init__(self):
        super(GaussianActivation, self).__init__()

    def forward(self, x):
        return torch.exp(-torch.pow(x, 2))

In [ ]:
class FeedforwardNN(torch.nn.Module):
    def __init__(self, input_size, output_size, mid_size_array, activation_flag):
        super(FeedforwardNN, self).__init__()

        self.input_size = input_size
        self.output_size = output_size
        self.mid_size_array = mid_size_array # Hidden layer sizes

        # Use ModuleList to hold a dynamic number of layers
        self.hidden_layers = torch.nn.ModuleList()

        # First layer
        self.hidden_layers.append(torch.nn.Linear(self.input_size, self.mid_size_array[0]))

        # Middle layers
        for i in range(len(self.mid_size_array) - 1):
            self.hidden_layers.append(torch.nn.Linear(self.mid_size_array[i], self.mid_size_array[i+1]))

        # Final layer
        self.output_layer = torch.nn.Linear(self.mid_size_array[-1], self.output_size)

        # activation
        if activation_flag == 'relu':
            self.activation = torch.nn.ReLU()
        elif activation_flag == 'sigmoid':
            self.activation = torch.nn.Sigmoid()
        elif activation_flag == 'softmax':
            self.activation = torch.nn.Softmax(dim=1) # dim=1 is standard for batched inputs
        elif activation_flag == 'gauss':
            self.activation = GaussianActivation()
        else:
            raise ValueError(f"Invalid activation_flag '{activation_flag}'. Choose 'relu', 'sigmoid', or 'softmax'.")

    def forward(self, x):
        # Iterate over all the hidden layers
        for layer in self.hidden_layers:
            # Pass data through the linear layer, then the activation function
            x = layer(x)
            x = self.activation(x)

        # Pass through the final output layer (usually no activation before loss computation)
        out = self.output_layer(x)

        return out


In [ ]:
def setup_model_and_data(x_train, y_train, K, mid_size_array, learning_rate=1.0, N_epochs=10000, activation_flag='sigmoid'): #use sigmoid by default

    # Dynamically grab input_size from data (e.g., 2 if x_train has 2 columns)
    # Alternatively, you can hardcode input_size = 2 if you prefer
    input_size = x_train.shape[1] if hasattr(x_train, 'shape') else print('x_train has no shape')
    output_size = K

    #Initialize the model
    model = FeedforwardNN(input_size, output_size, mid_size_array, activation_flag)

    #Store the input data as a PyTorch tensor
    x_train_tensor = torch.tensor(x_train, dtype=torch.float)

    #One hot encoding for labels
    y_onehot = np.zeros((y_train.size, K))
    y_onehot[np.arange(y_train.size), y_train] = 1
    y_onehot = torch.tensor(y_onehot, dtype=torch.float)

    #Setup Optimizer and Cost Function
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    cost_func = torch.nn.MSELoss()

    return model, x_train_tensor, y_onehot, optimizer, cost_func

In [ ]:
############ Function for plotting: ############
def updatePlot(model,epoch_list,cost_training,acc_training,x_train,y_train):

    ### Generate coordinates covering the whole plane: ###
    padding = 0.1
    spacing = 0.02
    x1_min, x1_max = x_train[:, 0].min() - padding, x_train[:, 0].max() + padding
    x2_min, x2_max = x_train[:, 1].min() - padding, x_train[:, 1].max() + padding
    x1_grid, x2_grid = np.meshgrid(np.arange(x1_min, x1_max, spacing),
                         np.arange(x2_min, x2_max, spacing))

    torch_input = torch.tensor(np.c_[x1_grid.ravel(), x2_grid.ravel()], dtype = torch.float)
    NN_output = model(torch_input)
    predicted_class = np.argmax(NN_output.detach().numpy(), axis=1)

    ### Plot the classifier: ###
    plt.subplot(121)
    plt.contourf(x1_grid, x2_grid, predicted_class.reshape(x1_grid.shape), K, alpha=0.8)
    plt.scatter(x_train[:, 0], x_train[:, 1], c=y_train, s=40)
    plt.xlim(x1_grid.min(), x1_grid.max())
    plt.ylim(x2_grid.min(), x2_grid.max())
    plt.xlabel(r'$x_1$')
    plt.ylabel(r'$x_2$')

    ### Plot the cost function during training: ###
    plt.subplot(222)
    plt.plot(epoch_list,cost_training,'o-')
    plt.xlabel('Epoch')
    plt.ylabel('Training cost')

    ### Plot the training accuracy: ###
    plt.subplot(224)
    plt.plot(epoch_list,acc_training,'o-')
    plt.xlabel('Epoch')
    plt.ylabel('Training accuracy')
############ End of plotting function ############

In [1]:
#reward function, tweak to determine what NN is optomizing
def rewardFN(acc_history,cost_history,test_accuracy, test_cost,cut_per):
    # Calculate the threshold index to slice the last quarter of the lists
    cutoff_index = int(len(acc_history) * (1-cut_per))

    # Convergence Speed
    # High mean accuracy means it reached high performance early
    acc_speed_metric = np.mean(acc_history)
    cost_speed_metric = np.mean(cost_history)

    # Smoothness Penalty
    # Calculate the first discrete difference of the cost curve
    acc_diffs = np.diff(acc_history[cutoff_index:])
    cost_diffs = np.diff(cost_history[cutoff_index:])
    # High standard deviation means highly volatile step-to-step changes
    acc_smoothness_penalty = np.std(acc_diffs)
    cost_smoothness_penalty = np.std(cost_diffs)

    # Base Achievement, use accuracy and cost from test set,
    # use std of diff of final 25% of acc and cost curves (want models which smoothly converge)
    # use mean of accuracy and cost curves, meaning models which quickly get accuracy and minimize cost are incentivized
    out = (
        test_accuracy - 0.35*test_cost
        - (0.1*acc_smoothness_penalty + 0.1*cost_smoothness_penalty)
        + (0.2 * acc_speed_metric - 0.1 * cost_speed_metric)
    )

    return out

In [ ]:
def evaluate_model(model, x_test, y_test, K, cost_func):
    """
    Evaluates a trained model on unseen test data.
    """
    # Convert test data to PyTorch tensors
    x_test_tensor = torch.tensor(x_test, dtype=torch.float)

    # One-hot encode y_test to calculate the test cost
    y_test_onehot = np.zeros((y_test.size, K))
    y_test_onehot[np.arange(y_test.size), y_test] = 1
    y_test_onehot = torch.tensor(y_test_onehot, dtype=torch.float)

    # Put the model into evaluation mode
    model.eval()

    # Turn off gradient calculation for inference (saves memory and compute)
    with torch.no_grad():
        NN_output = model(x_test_tensor)
        cost = cost_func(NN_output, y_test_onehot)

        # Calculate accuracy matching the training logic
        predicted_class = np.argmax(NN_output.numpy(), axis=1)
        accuracy = np.mean(predicted_class == y_test)

    return accuracy, cost.item()